In [1]:
import torch
torch.set_default_dtype(torch.double)
TOTAL = 20500
CENTER = 500
d = torch.randn(TOTAL, 2, requires_grad=True)
m = (-(d[:CENTER, None] - d[:CENTER]).square() / 2.0).sum(-1).exp()
print(torch.linalg.cond(m))
val, vec = torch.linalg.eigh(m)
print(
	torch.dist(vec @ val.diag_embed() @ vec.mH, m).item(),
	torch.dist(m @ vec @ (1.0 / val).diag_embed() @ vec.mH, torch.eye(CENTER)).item(),
	torch.dist(vec @ (1.0 / val).diag_embed() @ vec.mH @ m, torch.eye(CENTER)).item(),
	torch.linalg.cond(vec).item(),
	torch.linalg.cond(val.diag_embed()).item(),
	torch.linalg.cond(m @ vec).item(),
	torch.linalg.cond(vec.mH @ m).item()
)
ld, pivot, _ = torch.linalg.ldl_factor_ex(m)
print(
	torch.dist(torch.linalg.ldl_solve(ld, pivot, torch.eye(CENTER)) @ m, torch.eye(CENTER)).item(),
	torch.dist(m @ torch.linalg.ldl_solve(ld, pivot, torch.eye(CENTER)), torch.eye(CENTER)).item()
)

tensor(4.5745e+16, grad_fn=<SqueezeBackward1>)
6.274344717731801e-13 1602.4047071728753 2160.548149747603 1.0000000000000033 1.0007999171934436e+16 7.211137881866086e+16 4.093805064656529e+16
192554813.06334177 866.542273997613


In [2]:
import numpy as np
import sys
e = sys.float_info.epsilon / 2.0
A = torch.tensor([[e, 0.0],[0.0, 1.0]]) 
B = torch.tensor([1.0, 1.0 / e])

for driver in ['gels', 'gelsy', 'gelsd', 'gelss']:
	sols = [torch.linalg.lstsq(A, B, rcond=None, driver=driver)[0] for _ in range(10000)]
	errors = [((A @ sol - B)**2).sum().item() for sol in sols]
	print(f'{driver:5} max error {np.max(errors)} min error {np.min(errors)}')
	print(sols[np.argmax(errors)], sols[np.argmin(errors)])
print("")
for driver in ['gels', 'gelsy', 'gelsd', 'gelss']:
	sols = [torch.linalg.lstsq(A, B, rcond=0.0, driver=driver)[0] for _ in range(10000)]
	errors = [((A @ sol - B)**2).sum().item() for sol in sols]
	print(f'{driver:5} max error {np.max(errors)} min error {np.min(errors)}')
	print(sols[np.argmax(errors)], sols[np.argmin(errors)])

gels  max error 0.0 min error 0.0
tensor([9.0072e+15, 9.0072e+15]) tensor([9.0072e+15, 9.0072e+15])
gelsy max error 8.112963841460668e+31 min error 2.0
tensor([9.0072e+15, 0.0000e+00]) tensor([0.0000e+00, 9.0072e+15])
gelsd max error 1.0 min error 1.0
tensor([0.0000e+00, 9.0072e+15]) tensor([0.0000e+00, 9.0072e+15])
gelss max error 1.0 min error 1.0
tensor([0.0000e+00, 9.0072e+15]) tensor([0.0000e+00, 9.0072e+15])

gels  max error 0.0 min error 0.0
tensor([9.0072e+15, 9.0072e+15]) tensor([9.0072e+15, 9.0072e+15])
gelsy max error 2.0 min error 0.0
tensor([-0.0000e+00, 9.0072e+15]) tensor([9.0072e+15, 9.0072e+15])
gelsd max error 1.0 min error 1.0
tensor([0.0000e+00, 9.0072e+15]) tensor([0.0000e+00, 9.0072e+15])
gelss max error 0.0 min error 0.0
tensor([9.0072e+15, 9.0072e+15]) tensor([9.0072e+15, 9.0072e+15])


In [3]:
l = torch.tensor([[1, 1, 1], [e, 0, 0], [0, e, 0], [0, 0, e]])
print(l.T @ l)
print(torch.linalg.svd(l).S ** 2, torch.linalg.eigh(l.T @ l).eigenvalues)
print(torch.linalg.svdvals(l) ** 2, torch.linalg.eigvalsh(l.T @ l))

tensor([[1., 1., 1.],
        [1., 1., 1.],
        [1., 1., 1.]])
tensor([3.0000e+00, 1.2326e-32, 1.2326e-32]) tensor([-4.5198e-16, -1.5831e-17,  3.0000e+00])
tensor([3.0000e+00, 1.2326e-32, 1.2326e-32]) tensor([-5.8485e-16, -1.7700e-17,  3.0000e+00])


In [4]:
import time
m2 = (-(d[:, None] - d[:CENTER]).square() / 2.0).sum(-1).exp()
print(torch.linalg.cond(m2).item())
u, s, vh = torch.linalg.svd(m2)
print(
	torch.dist(m2, u @ torch.concat([torch.diag_embed(s), torch.zeros(TOTAL - CENTER, CENTER)], 0) @ vh).item() ** 2,
	torch.dist(torch.eye(CENTER), vh.mH @ torch.concat([torch.diag_embed(1.0 / s), torch.zeros(CENTER, TOTAL - CENTER)], 1) @ u.mH @ m2).item() ** 2,
	torch.dist(torch.eye(TOTAL), m2 @ vh.mH @ torch.concat([torch.diag_embed(1.0 / s), torch.zeros(CENTER, TOTAL - CENTER)], 1) @ u.mH).item() ** 2,
	torch.linalg.cond(m2 @ vh.mH @ (1.0 / s).diag_embed()).item()
)
ITER = 100
for driver in ['gels', 'gelss']:
	diff0 = np.empty(ITER)
	diff1 = np.empty(ITER)
	diff01 = np.empty(ITER)
	start = time.time()
	for i in range(ITER):
		sol0, res0, _, _ = torch.linalg.lstsq(m2, torch.eye(TOTAL), rcond=0.0, driver=driver)
		sol1, res1, _, _ = torch.linalg.lstsq(m2 @ vh.mH @ (1.0 / s).diag_embed(), torch.eye(TOTAL), rcond=0.0, driver=driver)
		diff0[i] = torch.dist(res0, (torch.eye(TOTAL) - m2 @ sol0).square().sum(-2)).item() ** 2
		diff1[i] = torch.dist(res1, (torch.eye(TOTAL) - m2 @ vh.mH @ (1.0 / s).diag_embed() @ sol1).square().sum(-2)).item() ** 2
		diff01[i] = torch.dist(res0, res1).item() ** 2
	print(
		f'{driver:5}',
		np.average(diff0),
		np.average(diff1),
		np.average(diff01),
		time.time() - start
	)

4.423330651146265e+16
8.386504052629632e-23 273.81213179643197 20331.415493202985 244.1113168667516
gels  1247.6826544215705 1.7540291610982952e-27 4.2076546481122055 1468.23885846138
gelss 31536.525270865983 1.7667544389567504e-27 4.2076546481122055 1688.343017578125
